# Network Routing Optimization
### Dijkstra · A\* · Bellman-Ford · Ant Colony Optimization on OSM Graphs

This notebook documents and runs the full routing optimization project built around OpenStreetMap data loaded via OSMnx.  
It covers:

1. **Environment setup** — imports, graph loading, projection  
2. **Algorithm implementations** — Dijkstra, A\*, Bellman-Ford, ACO (from `myAlgorithms.py`)  
3. **Single-city demo** — Times Square → Brooklyn Bridge on NYC  
4. **ACO diagnostics** — parameter tuning, convergence, and why ACO is a hybrid  
5. **Multi-city experiment** — 20-city benchmark (`run_AllCities.py` logic inline)  
6. **US scalability benchmark** — algorithm performance vs graph size (`benchmark_us.py` context)  
7. **GPS application overview** — how the algorithms power the Raspberry Pi navigator  
8. **Results analysis** — reading and visualising the CSVs produced by the experiments  

> **Prerequisites** — run the cell below to verify all packages are installed.

---
## 0  |  Environment & Imports

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import os
import sys
import csv
import math
import heapq
import time
import random
import statistics
import tracemalloc
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib

# ── Project module ────────────────────────────────────────────────────────────
# Make sure myAlgorithms.py is on the path.
# If this notebook lives in the same folder as myAlgorithms.py, this is automatic.
# Otherwise edit the path below.
sys.path.insert(0, str(Path(".").resolve()))
import myAlgorithms as ma

# ── Notebook display settings ─────────────────────────────────────────────────
%matplotlib inline
matplotlib.rcParams["figure.dpi"] = 130
plt.rcParams["figure.facecolor"] = "black"
plt.rcParams["axes.facecolor"]   = "black"
plt.rcParams["text.color"]       = "white"
plt.rcParams["axes.labelcolor"]  = "white"
plt.rcParams["xtick.color"]      = "white"
plt.rcParams["ytick.color"]      = "white"

print(f"OSMnx    version: {ox.__version__}")
print(f"NetworkX version: {nx.__version__}")
print("All imports OK ✓")

---
## 1  |  Load and Project the NYC Graph

The graph is loaded from a saved GraphML file using `ox.load_graphml()` (not `nx.read_graphml()`).  
Using OSMnx's loader preserves geometry, CRS, and typed attributes — all required for projection and pathfinding.

After loading we add `speed_kph` and `travel_time` edge attributes so all algorithms can use travel time as the weight.

In [ ]:
GRAPHML_PATH = "New York City_NY_USA.graphml"   # ← edit if your file is elsewhere

print("Loading graph …")
G = ox.load_graphml(GRAPHML_PATH)
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

print("Projecting graph …")
G_projected = ox.project_graph(G)
print("Graph projected ✓")

# ── Basic stats ───────────────────────────────────────────────────────────────
stats = ox.basic_stats(G_projected)
print(f"\nNodes              : {stats['n']:,}")
print(f"Edges              : {stats['m']:,}")
print(f"Avg degree (k_avg) : {stats['k_avg']:.2f}")
print(f"Street length (km) : {stats['street_length_total']/1000:.1f}")
print(f"Circuity avg       : {stats['circuity_avg']:.4f}")
print(f"Intersection count : {stats['intersection_count']:,}")

### What do the basic stats mean?

| Stat | Value | Meaning |
|------|-------|---------|
| `n` | 55,268 | Total nodes (intersections + dead ends) |
| `m` | 139,159 | Directed edges (2-way street = 2 edges) |
| `k_avg` | ~5.04 | Avg directed edges per node |
| `circuity_avg` | ~1.02 | Road length / straight-line distance — NYC's grid is 2% longer than straight |
| `intersection_count` | 51,644 | Nodes where ≥2 streets meet |

---
## 2  |  Select Origin and Destination Nodes

OSMnx node IDs are large integers drawn directly from OpenStreetMap.  
Use `ox.nearest_nodes()` on the **unprojected** graph (lat/lon) to find nodes — the projected graph shares the same IDs but stores coordinates in metres.

In [ ]:
# Coordinates are (longitude, latitude) — X=lon, Y=lat
TIMES_SQUARE    = {"lon": -73.9855, "lat": 40.7580, "name": "Times Square"}
BROOKLYN_BRIDGE = {"lon": -73.9969, "lat": 40.7061, "name": "Brooklyn Bridge"}

origin_node = ox.nearest_nodes(G, X=TIMES_SQUARE["lon"],    Y=TIMES_SQUARE["lat"])
dest_node   = ox.nearest_nodes(G, X=BROOKLYN_BRIDGE["lon"], Y=BROOKLYN_BRIDGE["lat"])

print(f"Origin node ({TIMES_SQUARE['name']})    : {origin_node}")
print(f"Dest   node ({BROOKLYN_BRIDGE['name']}) : {dest_node}")
print()

# Quick connectivity check
nx_path  = nx.shortest_path(G_projected, origin_node, dest_node, weight="travel_time")
nx_cost  = nx.shortest_path_length(G_projected, origin_node, dest_node, weight="travel_time")
print(f"NetworkX reference   : {nx_cost/60:.2f} min  |  {len(nx_path)} nodes")

---
## 3  |  Algorithm Implementations

All four algorithms live in `myAlgorithms.py`.  
The cells below run each one, verify correctness, and time them.

### 3.1  Dijkstra's Algorithm

**How it works:**
1. Start at origin with cost `0`; all others `∞`
2. Use a min-heap to always expand the cheapest unvisited node
3. Relax all neighbours — update if a cheaper path is found
4. Stop when the destination is popped from the heap

**Time complexity:** O((n + m) log n)

In [ ]:
t0 = time.perf_counter()
dijkstra_path, dijkstra_cost = ma.dijkstra(
    G_projected, origin_node, dest_node, weight="travel_time"
)
dijkstra_runtime = time.perf_counter() - t0

print(f"Dijkstra's")
print(f"  Cost    : {dijkstra_cost/60:.4f} min")
print(f"  Nodes   : {len(dijkstra_path)}")
print(f"  Runtime : {dijkstra_runtime:.4f}s")
print(f"  Matches NetworkX: {math.isclose(dijkstra_cost, nx_cost, rel_tol=1e-6)}")

### 3.2  A\* Algorithm

**How it works:**  
Same as Dijkstra's but every node's priority in the heap is `g(n) + h(n)` where:  
- `g(n)` = actual cost from origin to node n  
- `h(n)` = heuristic estimate from n to destination (Euclidean distance / avg speed)  

The heuristic guides the search spatially toward the destination, expanding fewer nodes than Dijkstra's.

**Time complexity:** O((n + m) log n) — but typically much faster in practice due to fewer node expansions.

In [ ]:
t0 = time.perf_counter()
astar_path, astar_cost = ma.astar(
    G_projected, origin_node, dest_node, weight="travel_time"
)
astar_runtime = time.perf_counter() - t0

print(f"A*")
print(f"  Cost    : {astar_cost/60:.4f} min")
print(f"  Nodes   : {len(astar_path)}")
print(f"  Runtime : {astar_runtime:.4f}s")
print(f"  Matches Dijkstra: {math.isclose(astar_cost, dijkstra_cost, rel_tol=1e-6)}")
print(f"  Speedup vs Dijkstra: {dijkstra_runtime/astar_runtime:.2f}x")

### 3.3  Bellman-Ford Algorithm

**How it works:**  
Relax **every edge** in the graph `n-1` times (where n = number of nodes).  
After k passes, all shortest paths using ≤ k edges are optimal.  
A final pass checks for negative cycles.

**Time complexity:** O(n × m) — significantly slower than Dijkstra's/A\*  
**Value:** Handles negative edge weights (not present in OSM) and serves as a correctness reference.

In [ ]:
print("Running Bellman-Ford (this may take a moment on large graphs) …")
t0 = time.perf_counter()
bf_path, bf_cost, has_neg_cycle = ma.bellman_ford(
    G_projected, origin_node, dest_node, weight="travel_time"
)
bf_runtime = time.perf_counter() - t0

print(f"\nBellman-Ford")
print(f"  Cost             : {bf_cost/60:.4f} min")
print(f"  Nodes            : {len(bf_path)}")
print(f"  Runtime          : {bf_runtime:.4f}s")
print(f"  Negative cycle   : {has_neg_cycle}")
print(f"  Matches Dijkstra : {math.isclose(bf_cost, dijkstra_cost, rel_tol=1e-6)}")

### 3.4  Correctness Verification

All three exact algorithms must agree on cost to within floating-point tolerance.

In [ ]:
print("=" * 50)
print("CORRECTNESS VERIFICATION")
print("=" * 50)
print(f"{'Algorithm':<15} {'Cost (min)':>12} {'Nodes':>8} {'Runtime':>12}")
print("-" * 50)
for name, cost, path, runtime in [
    ("Dijkstra's",    dijkstra_cost, dijkstra_path, dijkstra_runtime),
    ("A*",            astar_cost,    astar_path,    astar_runtime),
    ("Bellman-Ford",  bf_cost,       bf_path,       bf_runtime),
]:
    print(f"{name:<15} {cost/60:>12.4f} {len(path):>8} {runtime:>11.4f}s")

print()
print(f"Dijkstra == A*         : {math.isclose(dijkstra_cost, astar_cost, rel_tol=1e-6)}")
print(f"Dijkstra == Bellman-Ford: {math.isclose(dijkstra_cost, bf_cost, rel_tol=1e-6)}")
print(f"No negative cycles     : {not has_neg_cycle}")

---
## 4  |  ACO Diagnostics and Optimization

Ant Colony Optimization is a probabilistic algorithm — it does **not** guarantee the optimal path.  
Before running, we analyse the graph to recommend tuned parameters.

### 4.1  Pre-run Diagnostics

In [ ]:
# ── Quick pre-checks ──────────────────────────────────────────────────────────
test_path  = nx.shortest_path(G_projected, origin_node, dest_node, weight="travel_time")
duplicates = len(test_path) - len(set(test_path))
print(f"Dijkstra path length   : {len(test_path)} nodes")
print(f"Repeated nodes in path : {duplicates}")

G_sub     = ma.build_aco_subgraph(G_projected, origin_node, dest_node, padding=3000)
reachable = nx.single_source_shortest_path_length(G_sub, origin_node, cutoff=50)
print(f"Reachable within 50 hops: {len(reachable)}/{G_sub.number_of_nodes()} nodes")
print(f"Destination reachable   : {dest_node in reachable}")

In [ ]:
# ── Full parameter diagnostic ─────────────────────────────────────────────────
params = ma.diagnose_aco_parameters(
    G_projected, origin_node, dest_node,
    weight="travel_time", padding=3000
)
print("\nRecommended params:", params)

### 4.2  Run ACO

In [ ]:
t0 = time.perf_counter()
aco_path, aco_cost, aco_history = ma.ant_colony_optimization(
    G_projected,
    origin_node,
    dest_node,
    n_ants           = params["n_ants"],
    n_iterations     = params["n_iterations"],
    alpha            = params["alpha"],
    beta             = params["beta"],
    evaporation_rate = params["evaporation_rate"],
    deposit_weight   = params["deposit_weight"],
    weight           = "travel_time",
    padding          = 3000,
    max_steps        = params["max_steps"],
    seed_multiplier  = 10.0,
)
aco_runtime = time.perf_counter() - t0

if aco_cost < math.inf:
    gap = (aco_cost - dijkstra_cost) / dijkstra_cost * 100
    print(f"\nACO result  : {aco_cost/60:.2f} min  |  {len(aco_path)} nodes  |  {aco_runtime:.1f}s")
    print(f"Dijkstra ref: {dijkstra_cost/60:.2f} min")
    print(f"Gap         : {gap:.1f}% above optimal")
else:
    print("ACO failed to find a path.")

### 4.3  ACO Convergence Plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4), facecolor="black")
ax.set_facecolor("black")

valid = [(i, c / 60) for i, c in enumerate(aco_history) if c is not None]
if valid:
    iters, costs = zip(*valid)
    ax.plot(iters, costs, color="lime", linewidth=2, label="ACO best")

ax.axhline(dijkstra_cost / 60, color="cyan",   lw=1.5, ls="--",
           label=f"Dijkstra's ({dijkstra_cost/60:.2f} min)")
ax.axhline(astar_cost   / 60, color="orange", lw=1.5, ls=":",
           label=f"A*         ({astar_cost/60:.2f} min)")

ax.set_xlabel("Iteration")
ax.set_ylabel("Travel Time (min)")
ax.set_title("ACO Convergence vs Exact Algorithms", fontsize=13)
ax.legend(facecolor="#1a1a1a", labelcolor="white")
plt.tight_layout()
plt.show()

### 4.4  Why ACO is a Hybrid on OSM Graphs

Pure ACO uses only:
- **Pheromone** — reinforcement from previous ants
- **Heuristic** — local edge quality (1/cost)

Our implementation adds:
- **Directional bias** — spatial pull toward destination (borrowed from A\*)
- **Pheromone seeding** — initialised from Dijkstra's known optimal path

This is necessary because OSM road networks have low branching (~2.07 successors/node), directed one-way streets, and 4000+ nodes — making pure ACO extremely unlikely to find the destination by random walk alone.

| Property | Pure ACO | Hybrid ACO (this project) |
|----------|----------|---------------------------|
| Requires Dijkstra pre-run | No | Yes |
| Works on low-branching directed graphs | Poorly | Much better |
| Can find non-obvious paths | Yes | Less likely — biased toward Dijkstra corridor |
| Academically pure | Yes | No — but representative of real-world usage |

---
## 5  |  Route Visualisation — All Four Algorithms

In [ ]:
all_results = [
    (dijkstra_path, "cyan",    f"Dijkstra's\n{dijkstra_cost/60:.2f} min | {dijkstra_runtime:.4f}s"),
    (astar_path,    "orange",  f"A*\n{astar_cost/60:.2f} min | {astar_runtime:.4f}s"),
    (bf_path,       "magenta", f"Bellman-Ford\n{bf_cost/60:.2f} min | {bf_runtime:.4f}s"),
]
if aco_path:
    all_results.append(
        (aco_path, "lime", f"ACO (hybrid)\n{aco_cost/60:.2f} min | {aco_runtime:.1f}s")
    )

n_plots = len(all_results)
fig, axes = plt.subplots(1, n_plots, figsize=(8 * n_plots, 8), facecolor="black")
if n_plots == 1:
    axes = [axes]

# Plot Dijkstra first to get the natural zoom window
ox.plot_graph_route(
    G_projected, dijkstra_path,
    route_color="cyan", route_linewidth=3,
    node_size=0, bgcolor="black",
    ax=axes[0], show=False, close=False
)
axes[0].set_title(all_results[0][2], color="white", fontsize=11)
xlim = axes[0].get_xlim()
ylim = axes[0].get_ylim()

for i, (path, color, label) in enumerate(all_results[1:], start=1):
    ox.plot_graph_route(
        G_projected, path,
        route_color=color, route_linewidth=3,
        node_size=0, bgcolor="black",
        ax=axes[i], show=False, close=False
    )
    axes[i].set_title(label, color="white", fontsize=11)
    axes[i].set_xlim(xlim)
    axes[i].set_ylim(ylim)

plt.suptitle(
    f"{TIMES_SQUARE['name']} → {BROOKLYN_BRIDGE['name']} — Algorithm Comparison",
    color="white", fontsize=14
)
plt.tight_layout()
plt.savefig("algorithm_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6  |  Multi-City Experiment

This section mirrors `run_AllCities.py` — it runs all four algorithms across 20 cities, writes results to a CSV, and saves route images.

> **Note:** This will take several minutes to run and requires internet access to download graphs for any cities not already cached.

In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────────
TARGET_TRAVEL_TIME_SECONDS = 20 * 60
GRAPH_FOLDER   = "GraphML_Archive"
RESULTS_CSV    = "city_network_experiment_results.csv"
IMAGE_FOLDER   = "city_route_images"
RUN_ACO        = True   # set False to skip ACO for speed

CITIES = [
    "Cookeville, Tennessee, USA",
    "Lebanon, Tennessee, USA",
    "Murfreesboro, Tennessee, USA",
    "Chattanooga, Tennessee, USA",
    "Nashville, Tennessee, USA",
    "Charlotte, North Carolina, USA",
    "Raleigh, North Carolina, USA",
    "Atlanta, Georgia, USA",
    "Birmingham, Alabama, USA",
    "Louisville, Kentucky, USA",
]
# ↑ Trimmed to 10 for demo speed.  Restore all 20 from run_AllCities.py if desired.

graph_dir = Path(GRAPH_FOLDER)
image_dir = Path(IMAGE_FOLDER)
graph_dir.mkdir(exist_ok=True)
image_dir.mkdir(exist_ok=True)

print(f"Will run {len(CITIES)} cities.")
print(f"RUN_ACO = {RUN_ACO}")

In [ ]:
# ── Helper functions (inline versions of run_AllCities.py helpers) ────────────

def safe_filename(name):
    for ch in ['<', '>', ':', '"', '/', '\\', '|', '?', '*']:
        name = name.replace(ch, "_")
    return name.replace(", ", "_").replace(" ", "_")


def load_or_download_graph(place_name):
    fname      = safe_filename(place_name) + ".graphml"
    graph_path = graph_dir / fname
    if graph_path.exists():
        print(f"  Loading saved: {fname}")
        G = ox.load_graphml(graph_path)
    else:
        print(f"  Downloading: {place_name}")
        G = ox.graph_from_place(place_name, network_type="drive")
        ox.save_graphml(G, filepath=str(graph_path))
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)
    return G


def get_center_node(G):
    xs = [d["x"] for _, d in G.nodes(data=True)]
    ys = [d["y"] for _, d in G.nodes(data=True)]
    return ox.nearest_nodes(G, X=sum(xs)/len(xs), Y=sum(ys)/len(ys))


def choose_destination(G, origin_node):
    lengths, paths = nx.single_source_dijkstra(
        G, origin_node, cutoff=TARGET_TRAVEL_TIME_SECONDS * 1.75, weight="travel_time"
    )
    candidates = [n for n, c in lengths.items() if c >= TARGET_TRAVEL_TIME_SECONDS * 0.5]
    if not candidates:
        candidates = list(lengths.keys())
    dest = min(candidates, key=lambda n: abs(lengths[n] - TARGET_TRAVEL_TIME_SECONDS))
    return dest, paths[dest], lengths[dest]


def path_miles(G, path):
    total = 0
    for u, v in zip(path[:-1], path[1:]):
        e = min(G[u][v].values(), key=lambda e: e.get("length", float("inf")))
        total += e.get("length", 0)
    return total / 1609.34


def run_with_memory(func):
    tracemalloc.start()
    t0 = time.time()
    try:
        result = func()
        error  = ""
    except Exception as e:
        result = None
        error  = str(e)
    runtime = time.time() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, runtime, peak / (1024 * 1024), error


print("Helper functions defined ✓")

In [ ]:
# ── Main multi-city loop ──────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")   # non-interactive for image saving inside loop

rows = []

for city in CITIES:
    print(f"\n{'='*60}")
    print(f"CITY: {city}")
    print(f"{'='*60}")

    try:
        G_city      = load_or_download_graph(city)
        G_city_proj = ox.project_graph(G_city)
        origin      = get_center_node(G_city)
        dest, _, ref_time = choose_destination(G_city, origin)
        route_name  = city.split(",")[0]

        print(f"  Reference route: {ref_time/60:.2f} min")
        print(f"  Graph: {G_city.number_of_nodes():,} nodes")

        algorithms = [
            ("Dijkstra",     lambda: ma.dijkstra(G_city_proj, origin, dest, weight="travel_time")),
            ("A*",           lambda: ma.astar(G_city_proj, origin, dest, weight="travel_time")),
            ("Bellman-Ford", lambda: ma.bellman_ford(G_city_proj, origin, dest, weight="travel_time")),
        ]
        if RUN_ACO:
            algorithms.append(
                ("ACO", lambda: ma.ant_colony_optimization(
                    G_city_proj, origin, dest,
                    n_ants=30, n_iterations=80, alpha=1.0, beta=5.0,
                    evaporation_rate=0.15, deposit_weight=1000,
                    weight="travel_time", padding=5000, max_steps=1000,
                    seed_multiplier=10.0
                ))
            )

        for algo_name, algo_func in algorithms:
            result, runtime, memory_mb, error = run_with_memory(algo_func)

            if error:
                path, cost = [], math.inf
                has_neg    = ""
            elif algo_name == "Bellman-Ford":
                path, cost, has_neg = result
            elif algo_name == "ACO":
                path, cost, _ = result
                has_neg = ""
            else:
                path, cost = result
                has_neg = ""

            found = bool(path) and cost < math.inf

            row = {
                "City": city, "Algorithm": algo_name,
                "Found Path": found,
                "Travel Time Min": cost / 60 if found else "No path",
                "Runtime Seconds": round(runtime, 4),
                "Memory MB": round(memory_mb, 2),
                "Path Nodes": len(path),
                "Graph Nodes": G_city.number_of_nodes(),
                "Graph Edges": G_city.number_of_edges(),
                "Reference Time Min": round(ref_time / 60, 2),
                "Distance Miles": round(path_miles(G_city_proj, path), 2) if found else "No path",
                "Negative Cycle": has_neg,
                "Error": error,
            }
            rows.append(row)
            print(f"  {algo_name:<14}: {'✓' if found else '✗'}  "
                  f"{cost/60:.2f} min  {runtime:.3f}s  {memory_mb:.1f} MB")

            # Save route image
            if found:
                color_map = {"Dijkstra": "cyan", "A*": "orange",
                             "Bellman-Ford": "magenta", "ACO": "lime"}
                fig2, ax2 = ox.plot_graph_route(
                    G_city_proj, path,
                    route_color=color_map.get(algo_name, "red"),
                    route_linewidth=4, node_size=0,
                    bgcolor="white", show=False, close=False
                )
                ax2.set_title(f"{city} | {algo_name} | {cost/60:.2f} min", fontsize=9)
                img_path = image_dir / f"{safe_filename(route_name)}_{safe_filename(algo_name)}.png"
                fig2.savefig(img_path, dpi=120, bbox_inches="tight")
                plt.close(fig2)

    except Exception as e:
        print(f"  FAILED: {e}")
        rows.append({"City": city, "Algorithm": "CITY FAILED", "Error": str(e)})

# Save CSV
if rows:
    csv_path = Path(RESULTS_CSV)
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f"\n✓ Results saved to: {csv_path}")
    print(f"✓ Images saved to : {image_dir}/")

---
## 7  |  Analyse Multi-City Results

In [ ]:
%matplotlib inline

# Load CSV into a simple list of dicts
results_path = Path(RESULTS_CSV)
if not results_path.exists():
    print(f"Run Section 6 first to generate {RESULTS_CSV}")
else:
    with open(results_path) as f:
        city_data = list(csv.DictReader(f))

    # ── Runtime by algorithm (bar chart) ─────────────────────────────────────
    algo_names = ["Dijkstra", "A*", "Bellman-Ford", "ACO"]
    colors     = ["cyan", "orange", "magenta", "lime"]

    algo_runtimes = {a: [] for a in algo_names}
    for row in city_data:
        algo = row.get("Algorithm", "")
        rt   = row.get("Runtime Seconds", "")
        if algo in algo_runtimes and rt:
            try:
                algo_runtimes[algo].append(float(rt))
            except ValueError:
                pass

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="black")

    # Bar: avg runtime
    ax = axes[0]
    ax.set_facecolor("black")
    avgs   = [statistics.mean(algo_runtimes[a]) if algo_runtimes[a] else 0 for a in algo_names]
    bars   = ax.bar(algo_names, avgs, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_ylabel("Avg Runtime (s)")
    ax.set_title("Average Runtime by Algorithm", color="white")
    for bar, val in zip(bars, avgs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}s", ha="center", color="white", fontsize=9)

    # Scatter: runtime vs graph size (nodes)
    ax2 = axes[1]
    ax2.set_facecolor("black")
    for algo, color in zip(algo_names, colors):
        xs, ys = [], []
        for row in city_data:
            if row.get("Algorithm") == algo:
                try:
                    xs.append(int(row["Graph Nodes"]))
                    ys.append(float(row["Runtime Seconds"]))
                except (KeyError, ValueError):
                    pass
        if xs:
            ax2.scatter(xs, ys, color=color, label=algo, alpha=0.8, s=40)
    ax2.set_xlabel("Graph Nodes")
    ax2.set_ylabel("Runtime (s)")
    ax2.set_title("Runtime vs Graph Size", color="white")
    ax2.legend(facecolor="#1a1a1a", labelcolor="white")

    plt.suptitle("Multi-City Algorithm Benchmark", color="white", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 8  |  US Scalability Benchmark Overview

`benchmark_us.py` tests algorithm performance across **150 subgraphs** ranging from ~1,000 to 10,000,000 nodes,  
built from a filtered US OSM PBF file.

### How to Run the US Benchmark

```bash
# 1. Download the US OSM extract
#    https://download.geofabrik.de/north-america/us.html  → us-latest.osm.pbf

# 2. Filter to drivable roads only (requires osmium-tool)
osmium tags-filter us-latest.osm.pbf \
    w/highway=motorway,trunk,primary,secondary,tertiary,unclassified,residential \
    -o us_roads.osm.pbf --overwrite

# 3. Run the benchmark (produces src/results/benchmark_results.csv)
python benchmark_us.py
```

The cell below reads and visualises the benchmark CSV if it already exists.

In [ ]:
BENCHMARK_CSV = Path("src/results/benchmark_results.csv")

if not BENCHMARK_CSV.exists():
    print(f"{BENCHMARK_CSV} not found.")
    print("Run benchmark_us.py first (see instructions above).")
else:
    with open(BENCHMARK_CSV) as f:
        bm_rows = list(csv.DictReader(f))

    print(f"Loaded {len(bm_rows)} benchmark rows")

    # Parse out per-algo columns from the wide-format CSV
    algo_cols = {
        "Dijkstra":    {"runtime": "Runtime_s",   "cost": "Cost_Length"},
        "A*":          {"runtime": "Runtime_s",   "cost": "Cost_Length"},
        "Bellman-Ford":{"runtime": "Runtime_s",   "cost": "Cost_Length"},
        "ACO":         {"runtime": "Runtime_s",   "cost": "Cost_Length"},
    }

    # Build simple node_count → runtime per algo lists
    bm_nodes    = []
    bm_runtimes = {"Dijkstra": [], "A*": [], "Bellman-Ford": [], "ACO": []}

    for row in bm_rows:
        try:
            n = int(row["Num_Nodes"])
        except (KeyError, ValueError):
            continue
        bm_nodes.append(n)
        # Column names follow the HEADERS list in benchmark_us.py
        # (Dijkstra block starts at index 8, each block is 8 wide)
        keys = list(row.keys())
        for algo_idx, algo in enumerate(["Dijkstra", "A*", "Bellman-Ford", "ACO"]):
            rt_key = keys[8 + algo_idx * 8 + 5] if len(keys) > 8 + algo_idx * 8 + 5 else None
            try:
                bm_runtimes[algo].append(float(row[rt_key]))
            except (TypeError, ValueError, KeyError):
                bm_runtimes[algo].append(None)

    # Plot runtime vs node count
    fig, ax = plt.subplots(figsize=(13, 6), facecolor="black")
    ax.set_facecolor("black")

    for algo, color in zip(["Dijkstra", "A*", "Bellman-Ford", "ACO"],
                           ["cyan", "orange", "magenta", "lime"]):
        xs = [n for n, r in zip(bm_nodes, bm_runtimes[algo]) if r is not None]
        ys = [r for r in bm_runtimes[algo] if r is not None]
        if xs:
            ax.plot(xs, ys, color=color, label=algo, linewidth=1.5, alpha=0.85)

    ax.set_xlabel("Graph Nodes")
    ax.set_ylabel("Runtime (s)")
    ax.set_title("Runtime vs Graph Size — US Scalability Benchmark", fontsize=13)
    ax.legend(facecolor="#1a1a1a", labelcolor="white")
    ax.set_yscale("log")
    plt.tight_layout()
    plt.show()

---
## 9  |  GPS Application Overview

The routing algorithms from this project power a live navigation system running on a **Raspberry Pi Zero 2W**  
with a 7" touchscreen and a NEO-6M GPS module.

### File Structure

```
gps_app/
├── main.py        — Tkinter UI & application controller
├── gps_reader.py  — NEO-6M UART parser (with simulation fallback)
├── map_engine.py  — OSMnx graph loader & canvas renderer
├── routing.py     — All algorithms adapted for real-time routing
└── config.py      — All settings in one place
```

### Algorithm Selection in the GPS App

| Algorithm | GPS Use Case | Notes |
|-----------|-------------|-------|
| **A\*** | Default for all routes | Fastest real-world paths via Haversine heuristic |
| **Dijkstra's** | Long-distance routes | Guaranteed optimal; used when A\* is disabled |
| **Bellman-Ford** | Short local routes only | Too slow for large graphs; useful for edge case testing |
| **ACO** | Local routes only | Probabilistic; limited to graphs ≤ ~20k nodes |

### Live Re-routing
- GPS position is polled every second
- If deviation exceeds 50 m from the current route, the algorithm re-runs automatically
- A 15-second cooldown prevents thrashing on noisy GPS readings

In [ ]:
# ── Routing module summary ────────────────────────────────────────────────────
# This cell demonstrates how routing.py's find_path() dispatcher works.
# routing.py is the GPS app's algorithm interface — it wraps each algorithm
# with a consistent API: find_path(G, origin, dest, algorithm, weight) -> path

try:
    import routing
    print("routing.py loaded ✓")
    print("Available algorithms:", list(routing.ALGO_MAP.keys()))

    # Demo call — uses the NYC graph and nodes from Section 2
    demo_path = routing.find_path(
        G_projected, origin_node, dest_node,
        algorithm="A*", weight="travel_time"
    )
    print(f"\nrouting.find_path (A*): {len(demo_path)} nodes")
except ImportError:
    print("routing.py not found in current directory — skipping demo.")
    print("This module is part of the GPS app and not required for the benchmark.")

In [ ]:
# ── GPS simulation demo ───────────────────────────────────────────────────────
# Demonstrates what the GPSReader simulation mode produces without hardware.
import math

def simulate_gps_position(steps=10):
    """Mirror of GPSReader._simulate() — walks a small loop near Cookeville, TN."""
    base_lat, base_lon = 36.1628, -85.5016
    positions = []
    for step in range(steps):
        lat = base_lat + math.sin(step * 0.05) * 0.002
        lon = base_lon + math.cos(step * 0.05) * 0.002
        positions.append((lat, lon))
    return positions

sim_positions = simulate_gps_position(20)
print("Simulated GPS positions (lat, lon):")
for i, (lat, lon) in enumerate(sim_positions):
    print(f"  t={i:2d}s  ({lat:.6f}, {lon:.6f})")

---
## 10  |  Algorithm Complexity Summary

This final section summarises what we've learned about each algorithm's performance on OSM road graphs.

In [ ]:
%matplotlib inline

# ── Summary table ─────────────────────────────────────────────────────────────
summary = [
    ("Algorithm",     "Complexity",     "Optimal?", "OSM Suitable?", "Notes"),
    ("Dijkstra's",    "O((n+m) log n)", "Yes",      "Excellent",     "Best for large graphs"),
    ("A*",            "O((n+m) log n)", "Yes",      "Excellent",     "Faster than Dijkstra in practice"),
    ("Bellman-Ford",  "O(n × m)",       "Yes",      "Poor (slow)",   "Use only as correctness check"),
    ("ACO (hybrid)",  "O(iter×ants×steps)","No",   "Moderate",      "Needs Dijkstra seed + A* bias"),
]

print(f"{'Algorithm':<18} {'Complexity':<22} {'Optimal':>8} {'OSM Fit':>14} {'Notes'}")
print("-" * 85)
for row in summary[1:]:
    print(f"{row[0]:<18} {row[1]:<22} {row[2]:>8} {row[3]:>14}  {row[4]}")

print()

# ── Runtime comparison bar chart from NYC run ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4), facecolor="black")
ax.set_facecolor("black")

labels   = ["Dijkstra's", "A*", "Bellman-Ford", "ACO"]
runtimes = [dijkstra_runtime, astar_runtime, bf_runtime, aco_runtime]
colors   = ["cyan", "orange", "magenta", "lime"]

bars = ax.bar(labels, runtimes, color=colors, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, runtimes):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(runtimes) * 0.01,
        f"{val:.3f}s",
        ha="center", color="white", fontsize=10
    )

ax.set_ylabel("Runtime (seconds)")
ax.set_title("NYC Times Square → Brooklyn Bridge — Runtime Comparison", fontsize=12)
ax.set_yscale("log")
plt.tight_layout()
plt.show()

print(f"\nSpeedup A* vs Dijkstra's : {dijkstra_runtime/astar_runtime:.2f}x")
print(f"Slowdown Bellman-Ford vs A*: {bf_runtime/astar_runtime:.1f}x")

---
## 11  |  Key Takeaways

**Dijkstra's vs A\*:**  
Both are exact and produce identical paths. A\* is meaningfully faster because the Euclidean heuristic eliminates nodes far from the route — on NYC's grid this is a significant portion of the search space.

**Bellman-Ford:**  
Correct on all graphs including negative weights (none exist in OSM). Prohibitively slow at O(n×m) — only useful as a verification tool or in niche applications involving negative edge weights.

**ACO:**  
Fundamentally not suited to single-source single-destination routing on large directed OSM graphs. Low branching (~2 successors/node), one-way streets, and massive search space make convergence extremely slow without hybridisation. The hybrid (Dijkstra seed + A\* directional bias) makes it viable but it still cannot match exact algorithms on this problem type.

ACO's genuine strength is in **combinatorial problems** like Vehicle Routing (VRP) where Dijkstra's cannot give a direct answer — a natural next step for this project.

**GPS Application:**  
A\* is the clear default for real-time routing. The 15-second reroute cooldown and 50 m off-route threshold balance responsiveness with GPS noise tolerance on a resource-constrained Pi Zero 2W.